In [2]:
# !pip install scikit-learn==1.6.1
# !pip install xgboost==3.2.0
# !pip install optuna==4.8.0
# !pip install openpyxl==3.1.5

from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


# LASSO Regressor

In [ ]:
import json
import os
import re
import hashlib
import numpy as np
import pandas as pd
import optuna
from sklearn.model_selection import StratifiedGroupKFold
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso
import joblib

def save_results(rows, path):
    df_res = pd.DataFrame(rows)

    df_res = df_res.drop_duplicates(
        subset=["dataName", "seed","TW", "outer_fold"],
        keep="last"
    )
    df_res.to_excel(path, index=False)

def fit_outer_onehot(X_train_raw, X_test_raw, cat_cols):
    """
    Fit category levels from outer-train only, then one-hot outer-train and outer-test once.
    Returns aligned float32 matrices.
    """
    X_train_raw = X_train_raw.copy()
    X_test_raw = X_test_raw.copy()

    use_cat = [c for c in cat_cols if c in X_train_raw.columns]

    # lock category levels using OUTER TRAIN only
    for c in use_cat:
        train_cats = pd.Index(X_train_raw[c].dropna().unique())
        dtype = pd.CategoricalDtype(categories=train_cats, ordered=False)
        X_train_raw[c] = X_train_raw[c].astype(dtype)
        X_test_raw[c] = X_test_raw[c].astype(dtype)

    X_train_oh = pd.get_dummies(
        X_train_raw,
        columns=use_cat,
        drop_first=False
    )

    X_test_oh = pd.get_dummies(
        X_test_raw,
        columns=use_cat,
        drop_first=False
    )

    X_test_oh = X_test_oh.reindex(columns=X_train_oh.columns, fill_value=0)

    # keep exact same column order
    X_train_oh = X_train_oh.astype(np.float32)
    X_test_oh = X_test_oh.astype(np.float32)

    return X_train_oh, X_test_oh

for dataName in ["/content/drive/MyDrive/Colab_Backups/Data_3sMean_IDT_beforeMat.xlsx"]:

  for rs in [
      261,
              452,
              285,
              966,
            #  786, 1423, 3789, 5602, 8741, 11905
             ]:

      TW_comb = [[0,x] for x in range(1,31)]
      twType = "combineTW"
      # twType = "singleTW"

      # if rs == 0:
      #    TW_comb = [[0, x] for x in range(6,31)]
      # else:
      #    TW_comb = [[0, x] for x in range(1,31)]

      for TWs in TW_comb:

        print(TWs)

        if len(TWs)>1:

          targetTW = TWs[1]

        else:
          targetTW = TWs[0]

        corrThre = 0.85

        # ---------- Data Setup ----------
        df = pd.read_excel(dataName)

        BASE_DIR = "/content/drive/MyDrive/Colab_Backups/LRR"
        os.makedirs(BASE_DIR, exist_ok=True)

        results_path = os.path.join(BASE_DIR, f"LRR_{twType}.xlsx")
        modelDirectory = f"LR_model_{twType}"

        if os.path.exists(results_path):
              results_df = pd.read_excel(results_path)
              results_rows = results_df.to_dict('records')
              print(f"Resuming from existing file. Found {len(results_rows)} previous runs.")
        else:
              results_df = pd.DataFrame(columns=["seed", "TW"])
              results_rows = []

        # # ---- skip if this rs + TW already exists ----
        # if not results_df.empty:
        #     existing = (
        #         (results_df["seed"] == rs) &
        #         (results_df["TW"] == targetTW)
        #     )

        #     if existing.any():
        #         print(f"Skipping rs={rs}, targetTW={targetTW} because it already exists.")
        #         continue

        #   # ---- run your modelling code here ----
        # print(f"Running rs={rs}, targetTW={targetTW}")


        # ---------- Parameter ----------
        outer_splits, inner_splits, n_trials = 5, 4, 100

        target_col = "relativeRT"

        cat_names = ["Gender", "automation", "eventType"]

        TW_list = TWs

        feature_templates = [
            "Age",
            "License",
            "Mileage",
            "Gender",
            "automation", "eventType",
            "visibleSpeed",

            "pdMean_{TW}", "pdSd_{TW}",
            "blinkNumber_{TW}",
            "avgBlinkDuration_{TW}",

            "fixNumber_{TW}",
            "avgFixDuration_{TW}",

            "fixTE_{TW}", "fixSE_{TW}",
            "FixDisperX_{TW}", "FixDisperY_{TW}",
            "FixDisperE_{TW}",
            "FixMeanPosX_{TW}", "FixMeanPosY_{TW}",

        ]

        feature_cols = []
        for c in feature_templates:
            feature_cols.extend([c.format(TW=tw) for tw in TW_list] if "{TW}" in c else [c])

        cont_names = [c for c in feature_cols if c not in cat_names]


        tww_cols = [f"perValid_{i}" for i in range(31)]

        pervalid_mean = df[tww_cols].mean(axis=1)

        abandon_mask = pervalid_mean < 74.5
        keep_mask = ~abandon_mask

        df = df[keep_mask].copy()

        X = df[feature_cols].copy()
        y = pd.to_numeric(df[target_col], errors="coerce").values.astype(float)
        groups = df["subNumber"].values

        outer_cv = GroupKFold(
            n_splits=outer_splits,
            shuffle=True,
            random_state=rs
        )

        outer_scores = []
        outer_RMSE = []
        oof_pred = np.full_like(y, fill_value=-1)
        oof_fold = np.full(len(y), -1, dtype=int)
        oof_proba = np.full(len(y), np.nan)
        all_fold_best_vars = []

        outer_best_params = []
        outer_R2 = []
        fold_rows = []   # temp storage for this (dataName, rs, TW)

        print(f'len(df)={len(df)}')

        for outer_fold, (train_idx, test_idx) in enumerate(outer_cv.split(X, y, groups), start=1):


            print(f"\n=== Outer fold {outer_fold}/{outer_splits} ===")

            X_out_train = X.iloc[train_idx].copy()
            y_out_train = y[train_idx]
            groups_out_tr = groups[train_idx]

            X_out_test = X.iloc[test_idx].copy()
            y_out_test = y[test_idx]

            to_drop_fold = []

            for col in cat_names:
                if col in X_out_train.columns:
                    print(f"Fold {outer_fold} - {col} levels in train:",
                          X_out_train[col].unique())


            fold_vars = [c for c in feature_cols if c not in to_drop_fold]
            fold_cont = [c for c in fold_vars if c in cont_names]
            fold_cat = [c for c in fold_vars if c in cat_names]

            print(f"  > [TW-Filter] Dropped {len(to_drop_fold)} smaller windows. {len(fold_vars)} variables remain.")

            current_vars = fold_vars[:]
            best_vars = current_vars

            X_out_train_oh, X_out_test_oh = fit_outer_onehot(
                X_out_train[best_vars],
                X_out_test[best_vars],
                cat_names
            )


            def objective(trial):

                alpha = trial.suggest_float("alpha", 1e-4, 10.0, log=True)
                cv_opt = GroupKFold(
                    n_splits=inner_splits,
                    shuffle=True,
                    random_state=123
                )
                scores = []

                for t_i, v_i in cv_opt.split(X_out_train_oh, y_out_train, groups_out_tr):


                  X_tr = X_out_train_oh.iloc[t_i].copy()
                  X_val = X_out_train_oh.iloc[v_i].copy()
                  y_tr = y_out_train[t_i]
                  y_val = y_out_train[v_i]

                  # --- impute (fit ONLY on train fold) ---
                  imputer = SimpleImputer(strategy="median")
                  X_tr = pd.DataFrame(
                      imputer.fit_transform(X_tr),
                      columns=X_tr.columns,
                      index=X_tr.index
                  )
                  X_val = pd.DataFrame(
                      imputer.transform(X_val),
                      columns=X_val.columns,
                      index=X_val.index
                  )

                  # --- scale (fit ONLY on train fold) ---
                  scaler = StandardScaler()
                  X_tr = pd.DataFrame(
                      scaler.fit_transform(X_tr),
                      columns=X_tr.columns,
                      index=X_tr.index
                  )
                  X_val = pd.DataFrame(
                      scaler.transform(X_val),
                      columns=X_val.columns,
                      index=X_val.index
                  )

                  # --- model ---
                  model = Lasso(
                    alpha=alpha,
                    max_iter=5000,
                    random_state=42
                  )

                  model.fit(X_tr, y_tr)

                  y_pred = model.predict(X_val)

                  scores.append(np.sqrt(mean_squared_error(y_val, y_pred)))

                return float(np.mean(scores))


            sampler = optuna.samplers.TPESampler(seed=42)
            study = optuna.create_study(direction="minimize", sampler=sampler)
            study.optimize(objective, n_trials=n_trials)

            best_trial = study.best_trial

            # --- fit on OUTER TRAIN ONLY ---
            imputer = SimpleImputer(strategy="median")
            X_out_train_oh = pd.DataFrame(
                imputer.fit_transform(X_out_train_oh),
                columns=X_out_train_oh.columns,
                index=X_out_train_oh.index
            )
            X_out_test_oh = pd.DataFrame(
                imputer.transform(X_out_test_oh),
                columns=X_out_test_oh.columns,
                index=X_out_test_oh.index
            )

            scaler = StandardScaler()
            X_out_train_oh = pd.DataFrame(
                scaler.fit_transform(X_out_train_oh),
                columns=X_out_train_oh.columns,
                index=X_out_train_oh.index
            )
            X_out_test_oh = pd.DataFrame(
                scaler.transform(X_out_test_oh),
                columns=X_out_test_oh.columns,
                index=X_out_test_oh.index
            )

            final_p = {
                "alpha": study.best_params["alpha"],
                "max_iter": 5000,
                "random_state": 42
            }

            # --- final model ---
            final_model = Lasso(
                **final_p
            )

            outer_best_params.append(final_p)

            final_model.fit(X_out_train_oh, y_out_train)
            y_pred = final_model.predict(X_out_test_oh)


            oof_pred[test_idx] = y_pred
            oof_fold[test_idx] = outer_fold


            outer_rmse = np.sqrt(mean_squared_error(y_out_test, y_pred))

            r2   = r2_score(y_out_test, y_pred)

            fold_rows.append({
                "dataName": dataName,
                "seed": rs,
                "TW": float(targetTW),
                "outer_fold": int(outer_fold),
                "RMSE": float(outer_rmse),
                "R2": float(r2),
                "n_test": int(len(test_idx)),
            })

            plt.figure(figsize=(5, 5))
            plt.scatter(y_out_test, y_pred, alpha=0.6)
            outer_R2.append(r2)

            outer_scores.append(outer_rmse)

            print(f"{dataName} | Outer fold {outer_fold}/5 RMSE = {outer_rmse:.4f} R2 = {r2:.4f}")

            lims = [
                min(y_out_test.min(), y_pred.min()),
                max(y_out_test.max(), y_pred.max())
            ]
            plt.plot(lims, lims)
            plt.xlim(lims)
            plt.ylim(lims)

            plt.xlabel("Observed")
            plt.ylabel("Predicted")
            plt.title(
                f"Outer fold {outer_fold}\n"
                f"RMSE = {outer_rmse:.3f}, R² = {r2:.3f}"
            )

            plt.tight_layout()
            plt.show()

            MODEL_DIR = os.path.join(BASE_DIR, modelDirectory)
            os.makedirs(MODEL_DIR, exist_ok=True)

            model_path = os.path.join(MODEL_DIR, f"lr_seed_{rs:02d}_TW_{targetTW:02d}_outerfold_{outer_fold:02d}.joblib")
            joblib.dump(
                {
                    "model": final_model,
                    "imputer": imputer,
                    "scaler": scaler,
                    "feature_names": list(X_out_train_oh.columns)
                },
                model_path
            )
            print("Saved model:", model_path)


            meta = {
                "seed": rs,
                "TW": targetTW,
                "outer_fold": outer_fold,
                "best_params": final_p,
                "best_score_inner_cv": study.best_value,
                "selected_features": best_vars
            }

            meta_path = model_path.replace(".joblib", "_meta.json")

            with open(meta_path, "w") as f:
                json.dump(meta, f, indent=4)

            print("Saved metadata:", meta_path)


        # ---------- Final reporting ----------
        print(f"\n===== FINAL RESULTS SUMMARY =====")
        print(f"Mean RMSE (Outer CV): {np.mean(outer_scores):.4f} (+/- {np.std(outer_scores):.4f})")
        print(f"Mean R2 (Outer CV): {np.mean(outer_R2):.4f} (+/- {np.std(outer_R2):.4f})")


        # results_rows.extend(fold_rows)


        # summary_row = {
        #     "dataName": dataName,
        #     "seed": rs,
        #     "TW": float(targetTW),
        #     "outer_fold": "SUMMARY",
        #     "RMSE": float(np.mean(outer_scores)),
        #     "R2": float(np.mean(outer_R2)),
        #     "std_RMSE": float(np.std(outer_scores)),
        #     "std_R2": float(np.std(outer_R2)),
        #     "n_samples": int(len(y)),
        # }
        # results_rows.append(summary_row)

        # save_results(results_rows, results_path)
        # print(f"Updated results: {results_path}")

# LASSO Classifier

In [ ]:
import json
import os
import re
import hashlib
import numpy as np
import pandas as pd
import optuna
from sklearn.model_selection import StratifiedGroupKFold
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score, confusion_matrix
from sklearn.model_selection import GroupKFold
from sklearn.impute import SimpleImputer
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import joblib

def fit_winsor_caps(df: pd.DataFrame, cols, upper_q=0.95, lower_q=None):
    caps = {}
    for c in cols:
        x = pd.to_numeric(df[c], errors="coerce")
        lo = x.quantile(lower_q) if lower_q is not None else None
        hi = x.quantile(upper_q) if upper_q is not None else None
        caps[c] = (lo, hi)
    return caps


def apply_winsor_caps(df: pd.DataFrame, caps):
    out = df.copy()
    for c, (lo, hi) in caps.items():
        out[c] = pd.to_numeric(out[c], errors="coerce")
        if lo is not None:
            out[c] = out[c].clip(lower=lo)
        if hi is not None:
            out[c] = out[c].clip(upper=hi)
    return out

def fit_outer_onehot(X_train_raw, X_test_raw, cat_cols):
    """
    Fit category levels from outer-train only, then one-hot outer-train and outer-test once.
    Returns aligned float32 matrices.
    """
    X_train_raw = X_train_raw.copy()
    X_test_raw = X_test_raw.copy()

    use_cat = [c for c in cat_cols if c in X_train_raw.columns]

    # lock category levels using OUTER TRAIN only
    for c in use_cat:
        train_cats = pd.Index(X_train_raw[c].dropna().unique())
        dtype = pd.CategoricalDtype(categories=train_cats, ordered=False)
        X_train_raw[c] = X_train_raw[c].astype(dtype)
        X_test_raw[c] = X_test_raw[c].astype(dtype)

    X_train_oh = pd.get_dummies(
        X_train_raw,
        columns=use_cat,
        drop_first=False
    )

    X_test_oh = pd.get_dummies(
        X_test_raw,
        columns=use_cat,
        drop_first=False
    )

    X_test_oh = X_test_oh.reindex(columns=X_train_oh.columns, fill_value=0)

    # keep exact same column order
    X_train_oh = X_train_oh.astype(np.float32)
    X_test_oh = X_test_oh.astype(np.float32)

    return X_train_oh, X_test_oh

def evaluate_thresholds(y_true, y_prob, thresholds):
    best_thr = thresholds[0]
    best_score = -1

    for thr in thresholds:
        y_pred = (y_prob >= thr).astype(int)
        score = balanced_accuracy_score(y_true, y_pred)

        if score > best_score:
            best_score = score
            best_thr = thr

    return best_thr, best_score
def onehot_to_global(df_raw: pd.DataFrame, cat_cols, global_cols):
    cat_cols = [c for c in cat_cols if c in df_raw.columns]
    X = pd.get_dummies(df_raw, columns=cat_cols, drop_first=False)
    X = X.reindex(columns=global_cols, fill_value=0)
    return X.astype(float)

def ensure_categorical_dtype(df, cat_cols):
    """CRITICAL: Forces columns to 'category' so CatBoost uses native logic."""
    out = df.copy()
    for col in cat_cols:
        if col in out.columns:
            out[col] = out[col].astype("category")
    return out

# ---------- TW parsing ----------
def get_base_and_tw(col_name: str):
    match = re.match(r"(.+)_(\d+)$", col_name)
    if match:
        return match.group(1), int(match.group(2))
    return col_name, None


# ---------- Deterministic seed helper ----------
def stable_hash_int(s: str, mod: int = 10_000) -> int:
    return int(hashlib.md5(s.encode("utf-8")).hexdigest(), 16) % mod


def apply_imputer(df: pd.DataFrame, cont_cols, cat_cols, imp: SimpleImputer):
    out = df.copy()
    out[cont_cols] = out[cont_cols].apply(pd.to_numeric, errors="coerce")
    out[cat_cols] = out[cat_cols].astype("category")
    out[cont_cols] = imp.transform(out[cont_cols])
    return out

def save_results(rows, path):
    df_res = pd.DataFrame(rows)

    # Outer fold rows + summary row must be uniquely identified
    df_res = df_res.drop_duplicates(
        subset=["dataName", "seed","TW", "outer_fold"],
        keep="last"
    )
    df_res.to_excel(path, index=False)

def load_best_params(seed: int, TW: int, outer_fold: int):
    pattern = os.path.join(
        MODEL_DIR,
        f"xgb_seed_{seed:02d}_TW_{TW:02d}_outerfold_{outer_fold:02d}_meta.json"
    )
    if not os.path.exists(pattern):
        raise FileNotFoundError(f"Metadata not found: {pattern}")

    with open(pattern, "r") as f:
        meta = json.load(f)

    return meta["best_params"], meta.get("selected_features", None)

def load_best_params_RFE(seed: int, TW: int, outer_fold: int):
    pattern = os.path.join(
        MODEL_DIR,
        f"xgb_seed_{seed:02d}_TW_{TW:02d}_outerfold_{outer_fold:02d}_RFE_meta_RFE.json"
    )
    if not os.path.exists(pattern):
        raise FileNotFoundError(f"Metadata not found: {pattern}")

    with open(pattern, "r") as f:
        meta = json.load(f)

    return meta["best_params"], meta.get("selected_features", None)

def ensure_categorical_dtype_from_train(train_df, test_df, cat_cols):
    train_df = train_df.copy()
    test_df = test_df.copy()

    for col in cat_cols:
        if col in train_df.columns:
            cats = pd.Index(train_df[col].dropna().unique())
            dtype = pd.CategoricalDtype(categories=cats, ordered=False)
            train_df[col] = train_df[col].astype(dtype)
            test_df[col] = test_df[col].astype(dtype)

    return train_df, test_df

for dataName in ["/content/drive/MyDrive/Colab_Backups/Data_3sMean_IDT_beforeMat.xlsx"]:

    inner_seeds = pd.read_excel("/content/drive/MyDrive/Colab_Backups/inner_cv_seed_search_results_all10seeds.xlsx")
    inner_seeds_ok = inner_seeds[inner_seeds["accepted"] == True].copy()


    for rs in [
              3558, 6687,
               6836,
               7862, 11033,
               13285, 22075, 22163, 23858,
              30690
               ]:

        TW_comb = [[0,x] for x in range(1,31)]
        twType = "combineTW"
        # twType = "singleTW"

        # if rs == 261:
        #    TW_comb = [[0, x] for x in range(23,31)]
        # else:
        #    TW_comb = [[0, x] for x in range(1,31)]

        for TWs in TW_comb:

          print(TWs)

          if len(TWs)>1:

            targetTW = TWs[1]

          else:
            targetTW = TWs[0]

          corrThre = 0.85

          # ---------- Data Setup ----------
          df = pd.read_excel(dataName)

          BASE_DIR = "/content/drive/MyDrive/Colab_Backups/LR_test"
          os.makedirs(BASE_DIR, exist_ok=True)

          results_path = os.path.join(BASE_DIR, f"LRC_{twType}.xlsx")
          modelDirectory = f"LRC_model_{twType}"

          if os.path.exists(results_path):
              results_df = pd.read_excel(results_path)
              results_rows = results_df.to_dict('records')
              print(f"Resuming from existing file. Found {len(results_rows)} previous runs.")
          else:
              results_df = pd.DataFrame(columns=["seed", "TW"])
              results_rows = []

           # ---- skip if this rs + TW already exists ----
          if not results_df.empty:
              existing = (
                  (results_df["seed"] == rs) &
                  (results_df["TW"] == targetTW)
              )

              if existing.any():
                  print(f"Skipping rs={rs}, targetTW={targetTW} because it already exists.")
                  continue

          # ---- run your modelling code here ----
          print(f"Running rs={rs}, targetTW={targetTW}")


          # ---------- Parameter ----------
          outer_splits, inner_splits, n_trials = 5, 4, 100

          target_col = "relativeRT"

          cat_names = ["Gender", "automation", "eventType"]

          TW_list = TWs

          feature_templates = [
              "Age",
              "License",
              "Mileage",
              "Gender",
              "automation", "eventType",
              "visibleSpeed",

              "pdMean_{TW}", "pdSd_{TW}",
              "blinkNumber_{TW}",
              "avgBlinkDuration_{TW}",

              "fixNumber_{TW}",
              "avgFixDuration_{TW}",

              "fixTE_{TW}", "fixSE_{TW}",
              "FixDisperX_{TW}", "FixDisperY_{TW}",
              "FixDisperE_{TW}",
              "FixMeanPosX_{TW}", "FixMeanPosY_{TW}",

          ]

          feature_cols = []
          for c in feature_templates:
              feature_cols.extend([c.format(TW=tw) for tw in TW_list] if "{TW}" in c else [c])

          cont_names = [c for c in feature_cols if c not in cat_names]


          print(f'len(df)={len(df)}')

          df[target_col] = pd.to_numeric(df[target_col], errors="coerce")
          df = df.dropna(subset=[target_col, "eventType", "subNumber"]).copy()

          df["rt75_by_event"] = df.groupby("eventType")[target_col].transform(lambda x: x.quantile(0.85))
          df["RT75_cls"] = (df[target_col] >= df["rt75_by_event"]).astype(int)

          print(df[["eventType", "rt75_by_event"]].drop_duplicates().sort_values("eventType"))

          tww_cols = [f"perValid_{i}" for i in range(31)]

          pervalid_mean = df[tww_cols].mean(axis=1)

          abandon_mask = pervalid_mean < 74.5
          keep_mask = ~abandon_mask

          df = df[keep_mask].copy()

          X = df[feature_cols].copy()
          y = df["RT75_cls"].values.astype(int)

          groups = df["subNumber"].values

          y_strat = y.copy()


          outer_cv = StratifiedGroupKFold(
              n_splits=outer_splits,
              shuffle=True,
              random_state=rs
          )

          outer_scores = []
          outer_BA = []
          outer_AUC = []

          oof_pred = np.full_like(y, fill_value=-1)
          oof_fold = np.full(len(y), -1, dtype=int)
          oof_proba = np.full(len(y), np.nan)
          all_fold_best_vars = []

          outer_best_params = []

          fold_rows = []

          for outer_fold, (train_idx, test_idx) in enumerate(outer_cv.split(X, y_strat, groups), start=1):


              X_out_train = X.iloc[train_idx].copy()
              y_out_train = y[train_idx]
              groups_out_tr = groups[train_idx]

              X_out_test = X.iloc[test_idx].copy()
              y_out_test = y[test_idx]

              print(f"\n=== Outer fold {outer_fold}/{outer_splits} ===")

              print("Class counts:")
              print(pd.Series(y_out_test).value_counts().sort_index())
              print("Class proportions:")
              print(pd.Series(y_out_test).value_counts(normalize=True).sort_index())

              to_drop_fold = []

              for col in cat_names:
                  if col in X_out_train.columns:
                      print(f"Fold {outer_fold} - {col} levels in train:",
                            X_out_train[col].unique())


              fold_vars = [c for c in feature_cols if c not in to_drop_fold]
              fold_cont = [c for c in fold_vars if c in cont_names]
              fold_cat = [c for c in fold_vars if c in cat_names]

              print(f"  > [TW-Filter] Dropped {len(to_drop_fold)} smaller windows. {len(fold_vars)} variables remain.")

              current_vars = fold_vars[:]
              best_vars = current_vars

              n_pos = np.sum(y_out_train == 1)
              n_neg = np.sum(y_out_train == 0)
              scale_pos_weight = n_neg / n_pos if n_pos > 0 else 1.0

              print(f"Scale pos weight: {scale_pos_weight}")

              print(f"\nOuter fold {outer_fold}")
              print("Inner class counts:")
              print(pd.Series(y_out_train.copy()).value_counts().sort_index())
              print("Inner class proportions:")
              print(pd.Series(y_out_train.copy()).value_counts(normalize=True).sort_index())

              X_out_train_oh, X_out_test_oh = fit_outer_onehot(
                  X_out_train[best_vars],
                  X_out_test[best_vars],
                  cat_names
              )
              lookup = inner_seeds_ok[
                  (inner_seeds_ok["outer_seed"] == rs) &
                  (inner_seeds_ok["outer_fold_idx"] == (outer_fold - 1))
              ]

              if lookup.empty:
                  raise ValueError(
                      f"No accepted inner seed found for outer_seed={rs}, outer_fold_idx={outer_fold - 1}"
                  )

              if len(lookup) > 1:
                  raise ValueError(
                      f"Multiple matching inner seeds found for outer_seed={rs}, outer_fold_idx={outer_fold - 1}"
                  )

              selected_inner_seed = int(lookup["accepted_inner_seed"].iloc[0])

              print(
                  f"Using accepted_inner_seed={selected_inner_seed} "
                  f"for outer_seed={rs}, outer_fold={outer_fold}"
              )

              print("\nSelected inner seed:")
              print(selected_inner_seed)


              def objective(trial):

                  C = trial.suggest_float("C", 1e-4, 10.0, log=True)

                  cv_opt = StratifiedGroupKFold(
                      n_splits=inner_splits,
                      shuffle=True,
                      random_state=selected_inner_seed
                  )

                  inner_scores = []

                  for t_i, v_i in cv_opt.split(X_out_train_oh, y_out_train, groups_out_tr):


                    X_tr = X_out_train_oh.iloc[t_i].copy()
                    X_val = X_out_train_oh.iloc[v_i].copy()
                    y_tr = y_out_train[t_i]
                    y_val = y_out_train[v_i]

                    # --- impute (fit ONLY on train fold) ---
                    imputer = SimpleImputer(strategy="median")
                    X_tr = pd.DataFrame(
                        imputer.fit_transform(X_tr),
                        columns=X_tr.columns,
                        index=X_tr.index
                    )
                    X_val = pd.DataFrame(
                        imputer.transform(X_val),
                        columns=X_val.columns,
                        index=X_val.index
                    )

                    # --- scale (fit ONLY on train fold) ---
                    scaler = StandardScaler()
                    X_tr = pd.DataFrame(
                        scaler.fit_transform(X_tr),
                        columns=X_tr.columns,
                        index=X_tr.index
                    )
                    X_val = pd.DataFrame(
                        scaler.transform(X_val),
                        columns=X_val.columns,
                        index=X_val.index
                    )

                    # --- model ---
                    model = LogisticRegression(
                        penalty="l1",
                        solver="liblinear",
                        C=C,
                        max_iter=1000,
                        class_weight="balanced",
                        random_state=42,
                        # n_jobs=-1
                    )

                    model.fit(X_tr, y_tr)

                    y_prob = model.predict_proba(X_val)[:, 1]
                    inner_scores.append(roc_auc_score(y_val, y_prob))

                  return float(np.mean(inner_scores))


              sampler = optuna.samplers.TPESampler(seed=42)
              study = optuna.create_study(direction="maximize", sampler=sampler)
              study.optimize(objective, n_trials=n_trials)

              # --- fit on OUTER TRAIN ONLY ---
              imputer = SimpleImputer(strategy="median")
              X_out_train_oh = pd.DataFrame(
                  imputer.fit_transform(X_out_train_oh),
                  columns=X_out_train_oh.columns,
                  index=X_out_train_oh.index
              )
              X_out_test_oh = pd.DataFrame(
                  imputer.transform(X_out_test_oh),
                  columns=X_out_test_oh.columns,
                  index=X_out_test_oh.index
              )

              scaler = StandardScaler()
              X_out_train_oh = pd.DataFrame(
                  scaler.fit_transform(X_out_train_oh),
                  columns=X_out_train_oh.columns,
                  index=X_out_train_oh.index
              )
              X_out_test_oh = pd.DataFrame(
                  scaler.transform(X_out_test_oh),
                  columns=X_out_test_oh.columns,
                  index=X_out_test_oh.index
              )
              final_p = {
                  "penalty": "l1",
                  "solver": "liblinear",
                  "C": study.best_params["C"],
                  "max_iter": 1000,
                  "class_weight": "balanced",
                  "random_state": 42,
                  # "n_jobs": -1
              }

              # --- final model ---
              final_model = LogisticRegression(
                  **final_p
              )

              final_model.fit(X_out_train_oh, y_out_train)
              outer_best_params.append(final_p)
              y_prob_train = final_model.predict_proba(X_out_train_oh)[:, 1]

              thresholds = np.linspace(0.05, 0.95, 181)
              best_thr, best_ba = evaluate_thresholds(
                  y_out_train,
                  y_prob_train,
                  thresholds
              )

              print(f"Best threshold from outer train: {best_thr:.3f} | BA={best_ba:.4f}")

              # probabilities on OUTER TEST for evaluation
              y_prob = final_model.predict_proba(X_out_test_oh)[:, 1]
              y_pred = (y_prob >= best_thr).astype(int)

              oof_fold[test_idx] = outer_fold
              oof_proba[test_idx] = y_prob


              if np.unique(y_out_test).size == 2:
                  outer_auc = roc_auc_score(y_out_test, y_prob)
              else:
                  outer_auc = np.nan


              outer_AUC.append(outer_auc)
              # per-event AUC on outer test
              event_auc_dict = {}
              event_test = df.iloc[test_idx]["eventType"].values

              for ev in np.unique(event_test):
                  mask = event_test == ev

                  y_true_ev = y_out_test[mask]
                  y_prob_ev = y_prob[mask]

                  # AUC
                  if np.unique(y_true_ev).size == 2:
                      ev_auc = roc_auc_score(y_true_ev, y_prob_ev)
                  else:
                      ev_auc = np.nan

                  event_auc_dict[ev] = ev_auc

                  # use TRAIN-optimized threshold
                  y_pred_ev = (y_prob_ev >= best_thr).astype(int)

                  cm = confusion_matrix(y_true_ev, y_pred_ev, labels=[0,1])
                  tn, fp, fn, tp = cm.ravel()

                  ba = balanced_accuracy_score(y_true_ev, y_pred_ev)

                  print(f"  eventType={ev} | n={mask.sum()} | AUC={ev_auc:.4f}")
                  print(f"    thr={best_thr:.3f} | BA={ba:.4f}")
                  print(f"    CM -> TN={tn} FP={fp} FN={fn} TP={tp}")


              fold_row = {
                  "dataName": dataName,
                  "seed": rs,
                  "TW": float(targetTW),
                  "outer_fold": int(outer_fold),
                  "AUC": float(outer_auc),
                  "n_test": int(len(test_idx)),
                  "n_pos_test": int(np.sum(y_out_test == 1)),
                  "n_neg_test": int(np.sum(y_out_test == 0)),
              }

              # add per-event AUC columns
              for ev, ev_auc in event_auc_dict.items():
                  fold_row[f"AUC_event_{ev}"] = ev_auc

              fold_rows.append(fold_row)


              print(f"{dataName} | Outer fold {outer_fold}/5 AUC = {outer_auc:.4f}")


              # Save the model
              MODEL_DIR = os.path.join(BASE_DIR, modelDirectory)
              os.makedirs(MODEL_DIR, exist_ok=True)

              model_path = os.path.join(MODEL_DIR, f"lr_seed_{rs:02d}_TW_{targetTW:02d}_outerfold_{outer_fold:02d}.joblib")
              joblib.dump(
                  {
                      "model": final_model,
                      "imputer": imputer,
                      "scaler": scaler,
                      "feature_names": list(X_out_train_oh.columns)
                  },
                  model_path
              )
              print("Saved model:", model_path)


              # ---------- Save metadata ----------
              meta = {
                  "seed": rs,
                  "TW": targetTW,
                  "outer_fold": outer_fold,
                  "best_params": final_p,
                  "best_score_inner_cv": study.best_value,
                  "selected_features": best_vars
              }

              meta_path = model_path.replace(".joblib", "_meta.json")

              with open(meta_path, "w") as f:
                  json.dump(meta, f, indent=4)

              print("Saved metadata:", meta_path)


          # ---------- Final reporting ----------
          print(f"\n===== FINAL RESULTS SUMMARY =====")
          print(f"Mean AUC (Outer CV): {np.mean(outer_AUC):.4f} (+/- {np.std(outer_AUC):.4f})")

          results_rows.extend(fold_rows)

          fold_df = pd.DataFrame(fold_rows)

          event_auc_cols = [c for c in fold_df.columns if c.startswith("AUC_event_")]
          event_auc_summary = {f"mean_{c}": fold_df[c].mean(skipna=True) for c in event_auc_cols}
          event_auc_summary.update({f"std_{c}": fold_df[c].std(skipna=True) for c in event_auc_cols})


          summary_row = {
              "dataName": dataName,
              "seed": rs,
              "TW": float(targetTW),
              "outer_fold": "SUMMARY",
              "AUC": float(np.mean(outer_AUC)),
              "std_AUC": float(np.std(outer_AUC)),
              "n_samples": int(len(y)),
              "n_pos": int(np.sum(y == 1)),
              "n_neg": int(np.sum(y == 0)),
              **event_auc_summary
          }

          results_rows.append(summary_row)

          save_results(results_rows, results_path)
          print(f"Updated results: {results_path}")

Output hidden; open in https://colab.research.google.com to view.